In [1]:
import folium

print("Folium importado com sucesso!")

Folium importado com sucesso!


In [2]:
# Coordenadas de um trajeto já validado nos testes anteriores
origem = (-23.534377, -46.654322)
destino = (-23.525653, -46.679493)

mapa = folium.Map(
    location=[
        (origem[0] + destino[0]) / 2,
        (origem[1] + destino[1]) / 2
    ],
    zoom_start=14
)

folium.Marker(
    location=origem,
    popup="Origem informada",
    tooltip="Origem",
    icon=folium.Icon(color="green", icon="play")
).add_to(mapa)

folium.Marker(
    location=destino,
    popup="Destino informado",
    tooltip="Destino",
    icon=folium.Icon(color="red", icon="flag")
).add_to(mapa)

mapa

In [3]:
# Teste com quatro pontos diferentes

pontos_teste = [
    (
        "Origem do usuário (simulada)",
        (-23.53480, -46.65370),
        "green"
    ),
    (
        "Embarque - ANGELICA C/B",
        origem,
        "blue"
    ),
    (
        "Desembarque - PARADA PALMEIRAS C/B",
        destino,
        "purple"
    ),
    (
        "Destino do usuário (simulado)",
        (-23.52525, -46.68010),
        "red"
    )
]

def adicionar_marcadores(mapa):
    for nome, coordenadas, cor in pontos_teste:
        folium.Marker(
            location=coordenadas,
            popup=nome,
            tooltip=nome,
            icon=folium.Icon(color=cor)
        ).add_to(mapa)

    return mapa


mapa_pontos = folium.Map(
    location=origem,
    zoom_start=14
)

adicionar_marcadores(mapa_pontos)

mapa_pontos.fit_bounds(
    [coordenadas for _, coordenadas, _ in pontos_teste]
)

mapa_pontos

In [4]:
import io
import zipfile
import requests
import pandas as pd

url_gtfs = (
    "https://www.sptrans.com.br/umbraco/Surface/"
    "PerfilDesenvolvedor/BaixarGTFS"
)

# Baixar o arquivo GTFS
resposta = requests.get(url_gtfs, timeout=120)
resposta.raise_for_status()

trip_id_teste = "8000-1-1"

with zipfile.ZipFile(io.BytesIO(resposta.content)) as gtfs:

    # Localizar a viagem e seu shape_id
    viagens = pd.read_csv(
        gtfs.open("trips.txt"),
        dtype=str
    )

    viagem = viagens[
        viagens["trip_id"] == trip_id_teste
    ]

    if viagem.empty:
        raise ValueError(
            f"Viagem {trip_id_teste} não encontrada no GTFS atual."
        )

    shape_id = viagem.iloc[0]["shape_id"]

    # Recuperar somente os pontos desse trajeto
    partes = []

    for bloco in pd.read_csv(
        gtfs.open("shapes.txt"),
        usecols=[
            "shape_id",
            "shape_pt_lat",
            "shape_pt_lon",
            "shape_pt_sequence"
        ],
        dtype={"shape_id": str},
        chunksize=200000
    ):
        selecionados = bloco[
            bloco["shape_id"] == shape_id
        ]

        if not selecionados.empty:
            partes.append(selecionados)

if not partes:
    raise ValueError(
        f"Nenhum ponto encontrado para o shape_id {shape_id}."
    )

trajeto_gtfs = pd.concat(
    partes,
    ignore_index=True
).sort_values("shape_pt_sequence")

coordenadas_trajeto = list(
    zip(
        trajeto_gtfs["shape_pt_lat"],
        trajeto_gtfs["shape_pt_lon"]
    )
)

print("Viagem:", trip_id_teste)
print("Shape ID:", shape_id)
print("Quantidade de pontos:", len(coordenadas_trajeto))

Viagem: 8000-1-1
Shape ID: 70235
Quantidade de pontos: 193


In [6]:
# Criar mapa com percurso da linha de ônibus

mapa_trajeto = folium.Map(
    location=origem,
    zoom_start=13
)

# Adicionar os quatro marcadores
adicionar_marcadores(mapa_trajeto)

# Desenhar a geometria da linha
folium.PolyLine(
    locations=coordenadas_trajeto,
    color="blue",
    weight=5,
    opacity=0.8,
    tooltip=f"Percurso GTFS - {trip_id_teste}"
).add_to(mapa_trajeto)

# Ajustar visualização
mapa_trajeto.fit_bounds(
    coordenadas_trajeto
    + [coordenadas for _, coordenadas, _ in pontos_teste]
)

mapa_trajeto

In [7]:
from math import radians, sin, cos, sqrt, atan2

def distancia_metros(ponto1, ponto2):
    raio_terra = 6371000

    lat1, lon1 = map(radians, ponto1)
    lat2, lon2 = map(radians, ponto2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    return 2 * raio_terra * atan2(sqrt(a), sqrt(1 - a))


# Encontrar os pontos mais próximos das paradas
indice_embarque = min(
    range(len(coordenadas_trajeto)),
    key=lambda i: distancia_metros(
        origem,
        coordenadas_trajeto[i]
    )
)

indice_desembarque = min(
    range(len(coordenadas_trajeto)),
    key=lambda i: distancia_metros(
        destino,
        coordenadas_trajeto[i]
    )
)

dist_embarque = distancia_metros(
    origem,
    coordenadas_trajeto[indice_embarque]
)

dist_desembarque = distancia_metros(
    destino,
    coordenadas_trajeto[indice_desembarque]
)

print("Índice do embarque:", indice_embarque)
print("Índice do desembarque:", indice_desembarque)
print(f"Distância da parada ao traçado (embarque): {dist_embarque:.1f} m")
print(f"Distância da parada ao traçado (desembarque): {dist_desembarque:.1f} m")

if indice_embarque >= indice_desembarque:
    raise ValueError(
        "Os pontos encontrados não estão na ordem esperada do percurso."
    )

if max(dist_embarque, dist_desembarque) > 150:
    print("Atenção: uma parada ficou distante do traçado. Verifique o resultado.")

# Recortar o trecho da viagem
trecho_onibus = coordenadas_trajeto[
    indice_embarque:indice_desembarque + 1
]

print("Quantidade de pontos do trecho:", len(trecho_onibus))

Índice do embarque: 62
Índice do desembarque: 127
Distância da parada ao traçado (embarque): 4.4 m
Distância da parada ao traçado (desembarque): 14.7 m
Quantidade de pontos do trecho: 66


In [9]:
# Mapa com percurso completo e trecho recomendado

mapa_final_teste = folium.Map(
    location=origem,
    zoom_start=14
)

# Percurso completo em cinza
folium.PolyLine(
    locations=coordenadas_trajeto,
    color="gray",
    weight=4,
    opacity=0.5,
    tooltip="Percurso completo da linha"
).add_to(mapa_final_teste)

# Trecho que o passageiro utilizará, em azul
folium.PolyLine(
    locations=trecho_onibus,
    color="blue",
    weight=7,
    opacity=0.9,
    tooltip="Trecho recomendado de ônibus"
).add_to(mapa_final_teste)

# Os quatro marcadores
adicionar_marcadores(mapa_final_teste)

# Ajustar visualização para a viagem do passageiro
mapa_final_teste.fit_bounds(
    trecho_onibus
    + [coordenadas for _, coordenadas, _ in pontos_teste]
)

mapa_final_teste

mapa_final_teste.save("mapa_teste.html")

In [10]:
# TESTE 7 - Mapa completo da recomendação

# Recuperar os quatro pontos do teste anterior
local_origem = pontos_teste[0][1]
parada_embarque = pontos_teste[1][1]
parada_desembarque = pontos_teste[2][1]
local_destino = pontos_teste[3][1]

# Distâncias geográficas em linha reta
distancia_ate_embarque = distancia_metros(
    local_origem,
    parada_embarque
)

distancia_apos_desembarque = distancia_metros(
    parada_desembarque,
    local_destino
)

# Criar mapa
mapa_viagem = folium.Map(
    location=parada_embarque,
    zoom_start=14
)

# Percurso completo da linha (cinza)
folium.PolyLine(
    locations=coordenadas_trajeto,
    color="gray",
    weight=4,
    opacity=0.4,
    tooltip="Percurso completo da linha"
).add_to(mapa_viagem)

# Trecho recomendado de ônibus (azul)
folium.PolyLine(
    locations=trecho_onibus,
    color="blue",
    weight=7,
    opacity=0.9,
    tooltip="Trecho de ônibus recomendado"
).add_to(mapa_viagem)

# Ligação ilustrativa: origem até embarque
folium.PolyLine(
    locations=[local_origem, parada_embarque],
    color="green",
    weight=4,
    dash_array="6, 8",
    tooltip=(
        "Origem até embarque - "
        f"{distancia_ate_embarque:.0f} m em linha reta. "
        "Não representa rota real a pé."
    )
).add_to(mapa_viagem)

# Ligação ilustrativa: desembarque até destino
folium.PolyLine(
    locations=[parada_desembarque, local_destino],
    color="red",
    weight=4,
    dash_array="6, 8",
    tooltip=(
        "Desembarque até destino - "
        f"{distancia_apos_desembarque:.0f} m em linha reta. "
        "Não representa rota real a pé."
    )
).add_to(mapa_viagem)

# Adicionar os marcadores
adicionar_marcadores(mapa_viagem)

# Centralizar no trecho da viagem
mapa_viagem.fit_bounds(
    trecho_onibus
    + [coordenadas for _, coordenadas, _ in pontos_teste]
)

# Gerar HTML
mapa_viagem.save("mapa_viagem_teste.html")

print("Mapa gerado: mapa_viagem_teste.html")
print(f"Origem até embarque: {distancia_ate_embarque:.1f} m (linha reta)")
print(f"Desembarque até destino: {distancia_apos_desembarque:.1f} m (linha reta)")
print(
    "Distância total de aproximação: "
    f"{distancia_ate_embarque + distancia_apos_desembarque:.1f} m"
)

mapa_viagem

Mapa gerado: mapa_viagem_teste.html
Origem até embarque: 79.0 m (linha reta)
Desembarque até destino: 76.4 m (linha reta)
Distância total de aproximação: 155.4 m


In [11]:

def gerar_mapa_recomendacao(
    dados_gtfs,
    trip_id,
    origem_usuario,
    destino_usuario,
    parada_embarque,
    parada_desembarque,
    codigo_embarque,
    codigo_desembarque,
    nome_embarque,
    nome_desembarque,
    arquivo_html="mapa_recomendacao_teste.html"
):
    # 1. Localizar e validar a viagem no GTFS
    with zipfile.ZipFile(io.BytesIO(dados_gtfs)) as gtfs:

        viagens = pd.read_csv(
            gtfs.open("trips.txt"),
            dtype=str
        )

        viagem = viagens[viagens["trip_id"] == trip_id]

        if viagem.empty:
            raise ValueError(f"Viagem {trip_id} não encontrada.")

        shape_id = viagem.iloc[0]["shape_id"]

        # 2. Conferir a ordem das paradas
        registros = []

        for bloco in pd.read_csv(
            gtfs.open("stop_times.txt"),
            usecols=["trip_id", "stop_id", "stop_sequence"],
            dtype=str,
            chunksize=100000
        ):
            encontrados = bloco[
                (bloco["trip_id"] == trip_id)
                & bloco["stop_id"].isin([
                    str(codigo_embarque),
                    str(codigo_desembarque)
                ])
            ]

            if not encontrados.empty:
                registros.append(encontrados)

        if not registros:
            raise ValueError("Paradas não encontradas na viagem.")

        paradas_viagem = pd.concat(registros)

        embarques = paradas_viagem[
            paradas_viagem["stop_id"] == str(codigo_embarque)
        ]

        desembarques = paradas_viagem[
            paradas_viagem["stop_id"] == str(codigo_desembarque)
        ]

        if len(embarques) != 1 or len(desembarques) != 1:
            raise ValueError(
                "Paradas ausentes ou repetidas nesta viagem."
            )

        seq_embarque = int(embarques.iloc[0]["stop_sequence"])
        seq_desembarque = int(
            desembarques.iloc[0]["stop_sequence"]
        )

        if seq_embarque >= seq_desembarque:
            raise ValueError(
                "A ordem das paradas é incompatível com a viagem."
            )

        # 3. Recuperar a geometria da viagem
        partes = []

        for bloco in pd.read_csv(
            gtfs.open("shapes.txt"),
            usecols=[
                "shape_id",
                "shape_pt_lat",
                "shape_pt_lon",
                "shape_pt_sequence"
            ],
            dtype={"shape_id": str},
            chunksize=200000
        ):
            selecionados = bloco[
                bloco["shape_id"] == shape_id
            ]

            if not selecionados.empty:
                partes.append(selecionados)

    if not partes:
        raise ValueError("Geometria da viagem não encontrada.")

    geometria = pd.concat(partes, ignore_index=True)
    geometria = geometria.sort_values("shape_pt_sequence")

    percurso = list(zip(
        geometria["shape_pt_lat"],
        geometria["shape_pt_lon"]
    ))

    # 4. Localizar as paradas sobre a geometria
    indice_embarque = min(
        range(len(percurso)),
        key=lambda i: distancia_metros(
            parada_embarque, percurso[i]
        )
    )

    indice_desembarque = min(
        range(len(percurso)),
        key=lambda i: distancia_metros(
            parada_desembarque, percurso[i]
        )
    )

    distancia_shape_embarque = distancia_metros(
        parada_embarque, percurso[indice_embarque]
    )

    distancia_shape_desembarque = distancia_metros(
        parada_desembarque, percurso[indice_desembarque]
    )

    if indice_embarque >= indice_desembarque:
        raise ValueError(
            "O recorte da geometria não corresponde à viagem."
        )

    if max(
        distancia_shape_embarque,
        distancia_shape_desembarque
    ) > 150:
        raise ValueError(
            "Uma parada está muito distante da geometria."
        )

    trecho_onibus = percurso[
        indice_embarque:indice_desembarque + 1
    ]

    # 5. Criar o mapa
    mapa = folium.Map(
        location=parada_embarque,
        zoom_start=14
    )

    # Percurso completo
    folium.PolyLine(
        percurso,
        color="gray",
        weight=4,
        opacity=0.4,
        tooltip="Percurso completo da linha"
    ).add_to(mapa)

    # Trecho recomendado
    folium.PolyLine(
        trecho_onibus,
        color="blue",
        weight=7,
        tooltip=f"Trecho recomendado: {trip_id}"
    ).add_to(mapa)

    # Conexões ilustrativas em linha reta
    folium.PolyLine(
        [origem_usuario, parada_embarque],
        color="green",
        weight=4,
        dash_array="6,8",
        tooltip="Ligação ilustrativa, não rota real a pé"
    ).add_to(mapa)

    folium.PolyLine(
        [parada_desembarque, destino_usuario],
        color="red",
        weight=4,
        dash_array="6,8",
        tooltip="Ligação ilustrativa, não rota real a pé"
    ).add_to(mapa)

    # 6. Adicionar marcadores
    marcadores = [
        ("Origem", origem_usuario, "green"),
        (f"Embarque: {nome_embarque}", parada_embarque, "blue"),
        (f"Desembarque: {nome_desembarque}",
         parada_desembarque, "purple"),
        ("Destino", destino_usuario, "red")
    ]

    for nome, coordenadas, cor in marcadores:
        folium.Marker(
            location=coordenadas,
            popup=nome,
            tooltip=nome,
            icon=folium.Icon(color=cor)
        ).add_to(mapa)

    mapa.fit_bounds(
        trecho_onibus
        + [origem_usuario, destino_usuario]
        + [parada_embarque, parada_desembarque]
    )

    # 7. Salvar o HTML
    mapa.save(arquivo_html)

    print(f"Viagem: {trip_id}")
    print(f"Shape ID: {shape_id}")
    print(f"Sequência das paradas: {seq_embarque} → {seq_desembarque}")
    print(f"Pontos no trecho: {len(trecho_onibus)}")
    print(f"Arquivo gerado: {arquivo_html}")

    return mapa


In [14]:

mapa_dinamico = gerar_mapa_recomendacao(
    dados_gtfs=resposta.content,
    trip_id="8000-10-1",

    origem_usuario=(-23.53480, -46.65370),
    destino_usuario=(-23.52525, -46.68010),

    parada_embarque=(-23.534377, -46.654322),
    parada_desembarque=(-23.525653, -46.679493),

    codigo_embarque="7014416",
    codigo_desembarque="60016783",

    nome_embarque="ANGELICA C/B",
    nome_desembarque="PARADA PALMEIRAS C/B",

    arquivo_html="mapa_recomendacao_segundo_teste.html"
)

mapa_dinamico


Viagem: 8000-10-1
Shape ID: 70237
Sequência das paradas: 6 → 12
Pontos no trecho: 66
Arquivo gerado: mapa_recomendacao_segundo_teste.html
